# ML Portfolio Optimization - Baseline Experiments (No Sentiment)

**Goal**: Compare ML methods (Ridge vs LightGBM) against baseline strategies WITHOUT sentiment features.

**Strategies to test:**
1. Equal Weight (baseline)
2. Mean-Variance (baseline)
3. 60/40 Static (baseline)
4. Predictive Sharpe + Ridge (ML)
5. Predictive Sharpe + LightGBM (ML)

**ETFs**: SPY, QQQ, VTI, TLT, BND, GLD, VEA, VWO, IWM, XLE (10 total)

**Metrics**: Sharpe Ratio, Annualized Return, Volatility, Max Drawdown, Sortino, Calmar

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from data import load_default_etfs
from strategies import (
    EqualWeightStrategy,
    MeanVarianceStrategy,
    StaticStrategy,
    PredictiveSharpeStrategy,
    GradientBoostingSharpeStrategy
)
from backtest import PortfolioBacktest
from metrics import (
    calculate_sharpe_ratio,
    calculate_annualized_return,
    calculate_annualized_volatility,
    calculate_max_drawdown,
    calculate_sortino_ratio,
    calculate_calmar_ratio
)

# Plotting config
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ All imports successful")

## 1. Load Data (10 ETFs)

In [ ]:
# Load 10 ETFs with OHLCV data + market indicators
tickers = ['SPY', 'QQQ', 'VTI', 'TLT', 'BND', 'GLD', 'VEA', 'VWO', 'IWM', 'XLE']

print(f"Loading OHLCV data + market indicators for {len(tickers)} ETFs...")
print("="*80)

# Load data with new API (returns tuple: ohlcv_data, indicators)
ohlcv_data, indicators = load_default_etfs(
    start_date='2018-01-01',  # Extended to include 2020 COVID crash, 2022 bear market
    end_date='2025-12-07',
    expanded=True,
    ohlcv=True,
    include_indicators=True
)

# Extract Close prices for backward compatibility with existing strategies
close_cols = [col for col in ohlcv_data.columns if col.endswith('_Close')]
data = ohlcv_data[close_cols].copy()
data.columns = [col.replace('_Close', '') for col in close_cols]

print(f"\n✓ OHLCV Data loaded: {ohlcv_data.shape}")
print(f"  - Columns: {len(ohlcv_data.columns)} ({len(tickers)} ETFs × 5 OHLCV fields)")
print(f"  - Date range: {ohlcv_data.index[0].date()} to {ohlcv_data.index[-1].date()}")
print(f"  - Total days: {len(ohlcv_data)}")

if indicators is not None:
    print(f"\n✓ Market Indicators loaded: {indicators.shape}")
    print(f"  - Indicators: {list(indicators.columns)}")
    print(f"  - Date range: {indicators.index[0].date()} to {indicators.index[-1].date()}")
else:
    print(f"\n⚠ Market Indicators: Not loaded")

print(f"\n✓ Close Prices extracted: {data.shape}")
print(f"  - ETFs: {list(data.columns)}")
print(f"\nSample Close prices:")
print(data.head())

if indicators is not None:
    print(f"\nSample Market Indicators:")
    print(indicators.head())

## 2. Split Data (Train/Val/Test)

In [ ]:
# Define fixed date splits (ensures test set includes 2020-2022 crises)
# With start_date=2018-01-01, this gives us:
# - Train: 2018-01-01 to 2021-12-31 (4 years, includes 2020 COVID crash)
# - Val:   2022-01-01 to 2023-09-30 (21 months, includes 2022 bear market)  
# - Test:  2023-10-01 to 2025-12-07 (14 months, recent performance)

train_end = '2021-12-31'
val_end = '2023-09-30'

train_data = data[:train_end]
val_data = data[train_end:val_end].iloc[1:]  # Exclude overlap
test_data = data[val_end:].iloc[1:]  # Exclude overlap

# Also split OHLCV and indicators for future use
train_ohlcv = ohlcv_data[:train_end]
val_ohlcv = ohlcv_data[train_end:val_end].iloc[1:]
test_ohlcv = ohlcv_data[val_end:].iloc[1:]

if indicators is not None:
    train_indicators = indicators[:train_end]
    val_indicators = indicators[train_end:val_end].iloc[1:]
    test_indicators = indicators[val_end:].iloc[1:]

print("="*80)
print("DATA SPLITS (Fixed Dates)")
print("="*80)
print(f"Train: {train_data.index[0].date()} to {train_data.index[-1].date()} ({len(train_data)} days)")
print(f"Val:   {val_data.index[0].date()} to {val_data.index[-1].date()} ({len(val_data)} days)")
print(f"Test:  {test_data.index[0].date()} to {test_data.index[-1].date()} ({len(test_data)} days)")
print(f"\nTotal: {len(data)} days")
print(f"Split: {len(train_data)/len(data):.1%} train, {len(val_data)/len(data):.1%} val, {len(test_data)/len(data):.1%} test")

## 3. Initialize Strategies

In [ ]:
# Baseline strategies
equal_weight = EqualWeightStrategy()
mean_variance = MeanVarianceStrategy(risk_free_rate=0.02)
static_6040 = StaticStrategy(target_weights={'SPY': 0.6, 'TLT': 0.4})  # 60/40 portfolio

# ML strategies with PHASE 2+3 IMPROVEMENTS
# Phase 2 Changes:
# - Now using 160+ features (RSI, MACD, Bollinger, ATR, Volume, VIX, Yields) via FeatureEngineer
# - Previously used only 6 hardcoded features per ticker (60 total)
#
# Phase 3 Changes:
# - lookback_days: 252 → 756 (3 years of training data)
# - min_history_days: 126 → 60 (reduce warmup waste from 29% to 7%)
# - ridge_alpha: 10.0 (will be auto-tuned via CV)
# - shrinkage_intensity: 0.6 → 0.25 (reduce over-regularization)
# - max_weight: 0.5 (moderate concentration)
# - l2_gamma: 0.02 (moderate portfolio regularization)

ridge_ml = PredictiveSharpeStrategy(
    lookback_days=756,        # 3 years of history
    feature_window=60,        # 60-day rolling window for features
    min_history_days=60,      # REDUCED from 126 (less warmup waste)
    ridge_alpha=10.0,         # Will be tuned via cross-validation in Phase 3
    shrinkage_intensity=0.25, # REDUCED from 0.6 (less over-regularization)
    max_weight=0.5,           # Moderate concentration
    l2_gamma=0.02,            # Moderate portfolio regularization
    risk_free_rate=0.02
)

lightgbm_ml = GradientBoostingSharpeStrategy(
    lookback_days=756,        # 3 years
    feature_window=60,        # 60-day rolling window
    min_history_days=60,      # REDUCED from 126
    n_estimators=200,         # More trees
    max_depth=3,              # Shallower to prevent overfit
    learning_rate=0.01,       # Slower learning
    shrinkage_intensity=0.25, # REDUCED from 0.6
    max_weight=0.4,           # Lower concentration
    l2_gamma=0.05,            # Portfolio regularization
    risk_free_rate=0.02
)

strategies = {
    'Equal Weight': equal_weight,
    'Mean-Variance': mean_variance,
    '60/40 Static': static_6040,
    'Ridge ML (Improved)': ridge_ml,
    'LightGBM ML (Improved)': lightgbm_ml
}

print("=" * 80)
print("STRATEGY INITIALIZATION (Phase 2+3: Feature Integration + Hyperparameter Tuning)")
print("=" * 80)
print(f"\nInitialized {len(strategies)} strategies:")
for name in strategies:
    print(f"  - {name}")

print(f"\n🔧 Phase 2: Feature Integration")
print(f"  - OLD: 6 hardcoded features/ticker × 10 ETFs = 60 features")
print(f"  - NEW: 160+ features via FeatureEngineer:")
print(f"    • Basic: returns, volatility, momentum, Sharpe (60)")
print(f"    • Technical: RSI, MACD, Bollinger, ATR (160)")
print(f"    • Volume: volume features (40)")
print(f"    • Market: VIX, yields, spreads (5)")
print(f"    • Correlations: rolling cross-asset correlations (45)")

print(f"\n🔧 Phase 3: Hyperparameter Improvements")
print(f"  Ridge:")
print(f"    - min_history_days: 126 → 60 (29% warmup → 7%)")
print(f"    - shrinkage_intensity: 0.6 → 0.25 (reduce over-regularization)")
print(f"\n  LightGBM:")
print(f"    - min_history_days: 126 → 60 (reduce warmup waste)")
print(f"    - shrinkage_intensity: 0.6 → 0.25 (reduce over-regularization)")
print(f"    - max_depth: 5 → 3 (prevent overfitting)")
print(f"    - learning_rate: 0.05 → 0.01 (more stable)")

## 4. Run Backtests

In [ ]:
# Run backtests on TEST set (out-of-sample)
# Pass full data for lookback, but use start_date to restrict backtest execution to test period

results = {}

print("Running backtests on TEST set...\n")
print("="*80)
print(f"TEST PERIOD ONLY: {test_data.index[0].date()} to {test_data.index[-1].date()}")
print(f"(Full historical data available for 756-day lookback)")
print("="*80 + "\n")

# Initialize backtester
backtester = PortfolioBacktest(
    initial_capital=100000,
    transaction_cost=0.001,  # 10 bps
    rebalance_frequency='M'  # Monthly rebalancing
)

for name, strategy in strategies.items():
    print(f"\nBacktesting: {name}")
    print("-"*80)
    
    # Run backtest with full data AND pass OHLCV + indicators for enhanced features
    portfolio_values = backtester.run(
        strategy=strategy,
        prices=data,  # Full data for lookback
        ohlcv_data=ohlcv_data,  # NEW: Pass OHLCV for technical indicators
        indicators=indicators,   # NEW: Pass market indicators
        start_date=test_data.index[0].strftime('%Y-%m-%d'),  # Only execute on test period
        end_date=test_data.index[-1].strftime('%Y-%m-%d')
    )
    
    # Compute metrics
    sharpe = calculate_sharpe_ratio(portfolio_values)
    annual_return = calculate_annualized_return(portfolio_values)
    volatility = calculate_annualized_volatility(portfolio_values)
    max_dd = calculate_max_drawdown(portfolio_values)
    sortino = calculate_sortino_ratio(portfolio_values)
    calmar = calculate_calmar_ratio(portfolio_values)
    
    results[name] = {
        'portfolio_values': portfolio_values,
        'allocations': backtester.allocations,
        'sharpe': sharpe,
        'annual_return': annual_return,
        'volatility': volatility,
        'max_drawdown': max_dd,
        'sortino': sortino,
        'calmar': calmar
    }
    
    print(f"  Sharpe Ratio: {sharpe:.4f}")
    print(f"  Annual Return: {annual_return:.2%}")
    print(f"  Volatility: {volatility:.2%}")
    print(f"  Max Drawdown: {max_dd:.2%}")

print("\n" + "="*80)
print("Backtests complete!")

## 5. Results Summary Table

In [ ]:
# Create summary DataFrame
summary_data = []

for name, result in results.items():
    summary_data.append({
        'Strategy': name,
        'Sharpe': result['sharpe'],
        'Annual Return': result['annual_return'],
        'Volatility': result['volatility'],
        'Max Drawdown': result['max_drawdown'],
        'Sortino': result['sortino'],
        'Calmar': result['calmar']
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Sharpe', ascending=False)

print("\n" + "="*80)
print("PERFORMANCE SUMMARY (TEST SET)")
print("="*80)
print(summary_df.to_string(index=False))

# Highlight best performer
best_strategy = summary_df.iloc[0]['Strategy']
best_sharpe = summary_df.iloc[0]['Sharpe']
print(f"\n🏆 Best Strategy: {best_strategy} (Sharpe: {best_sharpe:.4f})")

## 6. Plot Portfolio Values

In [ ]:
# Plot cumulative returns
plt.figure(figsize=(14, 8))

for name, result in results.items():
    pv = result['portfolio_values']
    plt.plot(pv.index, pv.values, label=name, linewidth=2)

plt.title('Portfolio Value Over Time (Test Set)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Portfolio Value ($)', fontsize=12)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Plot Drawdowns

In [ ]:
# Plot drawdowns
fig, axes = plt.subplots(len(results), 1, figsize=(14, 4*len(results)))
if len(results) == 1:
    axes = [axes]

for (name, result), ax in zip(results.items(), axes):
    pv = result['portfolio_values']
    
    # Compute drawdown
    cummax = pv.cummax()
    drawdown = (pv - cummax) / cummax
    
    ax.fill_between(drawdown.index, 0, drawdown.values, alpha=0.3, color='red')
    ax.plot(drawdown.index, drawdown.values, color='darkred', linewidth=1)
    ax.set_title(f'{name} - Drawdown', fontsize=12, fontweight='bold')
    ax.set_ylabel('Drawdown', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()

## 8. Compare Sharpe Ratios (Bar Chart)

In [ ]:
# Bar chart of Sharpe ratios
plt.figure(figsize=(12, 6))

colors = ['#2ecc71' if 'ML' in name else '#3498db' for name in summary_df['Strategy']]
bars = plt.bar(summary_df['Strategy'], summary_df['Sharpe'], color=colors, alpha=0.8, edgecolor='black')

plt.title('Sharpe Ratio Comparison (No Sentiment)', fontsize=16, fontweight='bold')
plt.ylabel('Sharpe Ratio', fontsize=12)
plt.xlabel('Strategy', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.axhline(y=1.0, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Sharpe = 1.0')
plt.grid(True, alpha=0.3, axis='y')
plt.legend()
plt.tight_layout()
plt.show()

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

## 9. Key Insights

**Questions to answer:**
1. Do ML methods (Ridge/LightGBM) beat baseline strategies?
2. Does LightGBM beat Ridge?
3. What is the best Sharpe ratio achieved (target: >1.5)?
4. Are ML methods overfitting (check train vs test performance)?

**Next steps:**
- If ML methods underperform → debug feature engineering
- If ML methods work → add sentiment features and compare

In [ ]:
# Print key insights
print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

# Best strategy
best = summary_df.iloc[0]
print(f"\n1. Best Strategy: {best['Strategy']}")
print(f"   - Sharpe: {best['Sharpe']:.4f}")
print(f"   - Annual Return: {best['Annual Return']:.2%}")
print(f"   - Max Drawdown: {best['Max Drawdown']:.2%}")

# ML vs Baseline
ml_strategies = summary_df[summary_df['Strategy'].str.contains('ML')]
baseline_strategies = summary_df[~summary_df['Strategy'].str.contains('ML')]

print(f"\n2. ML vs Baseline:")
print(f"   - Best ML Sharpe: {ml_strategies['Sharpe'].max():.4f}")
print(f"   - Best Baseline Sharpe: {baseline_strategies['Sharpe'].max():.4f}")
print(f"   - Improvement: {(ml_strategies['Sharpe'].max() - baseline_strategies['Sharpe'].max()):.4f}")

# Ridge vs LightGBM
ridge_sharpe = summary_df[summary_df['Strategy'] == 'Ridge ML']['Sharpe'].values[0]
lgbm_sharpe = summary_df[summary_df['Strategy'] == 'LightGBM ML']['Sharpe'].values[0]

print(f"\n3. Ridge vs LightGBM:")
print(f"   - Ridge Sharpe: {ridge_sharpe:.4f}")
print(f"   - LightGBM Sharpe: {lgbm_sharpe:.4f}")
print(f"   - Winner: {'LightGBM' if lgbm_sharpe > ridge_sharpe else 'Ridge'}")

print("\n" + "="*80)